<a href="https://colab.research.google.com/github/jooyongsim/deeplearning_SMWU_2026/blob/main/DL_assignment01/client/RopeT_Arena_Client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rope-T Arena · team client (Colab)

This notebook connects **your team's trained policy** to the Rope-T Arena on game day.
The arena runs the simulator and sends your policy the 85-number observation each step; your policy answers with an action.

**You need:**
1. `il.zip`: a zip of your team's `il/` folder (your `policy.py`, `dataset.py`, …)
2. your checkpoint, e.g. `runs/m3/ckpt.pt`
3. the arena address, your team id (`team1` … `team5`) and your key, given by the host

Run the cells top to bottom. **Keep this tab open during the game**: Colab disconnects idle sessions.
A laptop with your assignment environment works the same way and is the recommended option (see the arena's *How it works* page).

## 1 · Arena settings (fill in)

In [ ]:
ARENA_URL = "https://soon-databases-displays-rogers.trycloudflare.com"   # the address shown on the projector
TEAM      = "team1"                                   # your team id
KEY       = "REPLACE-ME"                              # your team's key (keep it secret)
NOTE      = "BC chunk"                                # shown to the host, e.g. "BC chunk + PPO"

## 2 · Install and get the assignment code

In [ ]:
!pip -q install mujoco==3.3.1 "websockets>=12"
import os
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/jooyongsim/deeplearning_SMWU_2026.git /content/repo
%cd /content/repo/DL_assignment01
import sys, urllib.request
src = urllib.request.urlopen(ARENA_URL.rstrip('/') + '/client/arena_client.py', timeout=30).read().decode()
assert 'def load_policy' in src, 'did not get arena_client.py: check ARENA_URL and that the arena is running, then rerun this cell'
open('arena_client.py', 'w').write(src); sys.modules.pop('arena_client', None)   # forget any earlier (broken) import
print('arena_client.py downloaded from', ARENA_URL)

## 3 · Upload your `il.zip` and your checkpoint
Select both files in the upload dialog. Your `il/` replaces the template's.

In [ ]:
import zipfile, shutil, glob
from google.colab import files
up = files.upload()
zips = [f for f in up if f.endswith('.zip')]; ckpts = [f for f in up if f.endswith('.pt')]
assert zips and ckpts, 'upload il.zip and your .pt checkpoint'
with zipfile.ZipFile(zips[0]) as z:
    names = z.namelist(); z.extractall('/content/il_up')
src = '/content/il_up/il' if os.path.isdir('/content/il_up/il') else '/content/il_up'
for f in glob.glob(src + '/*.py'): shutil.copy(f, 'il/')
CKPT = ckpts[0]
print('il/ updated with', sorted(os.path.basename(f) for f in glob.glob(src + '/*.py')), '| checkpoint:', CKPT)

## 4 · Load your policy and check it answers

In [ ]:
%cd /content/repo/DL_assignment01
import json, urllib.request, glob, os
import numpy as np
if 'CKPT' not in globals():          # e.g. after a runtime restart: the uploaded files are still on disk
    pts = sorted(glob.glob('*.pt'), key=os.path.getmtime)
    assert pts, 'no checkpoint found: run the upload cell (3) first'
    CKPT = pts[-1]; print('using the checkpoint uploaded earlier:', CKPT)
from arena_client import load_policy, play
agent, info = load_policy(repo='.', ckpt=CKPT, note=NOTE)
print('params', info['params'], '| sha256', info['sha256'][:16])
obs0 = np.array(json.load(urllib.request.urlopen(ARENA_URL.rstrip('/') + '/api/sample_obs', timeout=20))['obs'], np.float32)
agent.reset(); a = agent.act(obs0)
print('action for a sample observation:', np.round(a, 3))
assert np.shape(a) == (3,) and np.all(np.abs(a) <= 1), 'the policy must return 3 numbers in [-1, 1]'

## 5 · Enter the arena
This cell keeps running and plays every round the host starts. Your lane on the projector turns **online**. Stop it with the ■ button after the game.

In [ ]:
await play(ARENA_URL, TEAM, KEY, agent, info)